# 贷款违约风险分析

## 项目背景
基于贷款数据集（和鲸社区获取），分析借款人特征与违约风险的关系，并构建简单的风险预测规则。

## 数据说明
- 数据源：loandata.xls（含年龄、收入、负债率、信用卡负债、教育、违约等字段）
- 工具：Python + pandas

## 分析流程
1. 数据读取与质量检测（空值、重复值）
2. 分组聚合分析（按教育、年龄段统计收入/负债/违约情况）
3. 规则模型构建（基于收入与负债率的违约预测）
4. 模型评估（正确率计算）
5. 透视表与索引操作练习


In [1]:
# ===== 导入库 =====
# pandas：数据分析核心库
import pandas as pd
import numpy as np

In [7]:
# ===== 读取数据 =====
# 读取贷款数据集 Excel 文件（注意：绝对路径，换环境需修改）
Loan_data = pd.read_excel('D:/浏览器杂项/贷款违约预测/loandata/loandata.xls')
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约
0,41,3,17,12,176,9.3,11.359392,5.008608,1
1,27,1,10,6,31,17.3,1.362202,4.000798,0
2,40,1,15,14,55,5.5,0.856075,2.168925,0
3,41,1,15,14,120,2.9,2.658720,0.821280,0
4,24,2,2,0,28,17.3,1.787436,3.056564,1
...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,1
696,29,2,6,4,21,11.5,0.369495,2.045505,0
697,33,1,15,3,32,7.6,0.491264,1.940736,0
698,45,1,19,22,77,8.4,2.302608,4.165392,0


In [8]:
# ⚠️ 变量名错误：定义的是 Loan_data（大写L），此处是 loan_data（小写l），运行会 NameError
loan_data

NameError: name 'loan_data' is not defined

In [7]:
# 查看前 5 行数据
Loan_data.head(5)

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约
0,41,3,17,12,176,9.3,11.359392,5.008608,1
1,27,1,10,6,31,17.3,1.362202,4.000798,0
2,40,1,15,14,55,5.5,0.856075,2.168925,0
3,41,1,15,14,120,2.9,2.658720,0.821280,0
4,24,2,2,0,28,17.3,1.787436,3.056564,1


In [8]:
# 查看后 5 行数据
Loan_data.tail(5)

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约
695,36,2,6,15,27,4.6,0.262062,0.979938,1
696,29,2,6,4,21,11.5,0.369495,2.045505,0
697,33,1,15,3,32,7.6,0.491264,1.940736,0
698,45,1,19,22,77,8.4,2.302608,4.165392,0
699,37,1,12,14,44,14.7,2.994684,3.473316,0


In [9]:
# ===== 数据质量检测 =====
# isnull().sum()：各列缺失值数量；duplicated().sum()：重复行数量
print('<<<<<<<<<<<<<<空值和重复值检测>>>>>>>>>>>>>')
print('空值检测结果：',Loan_data.isnull().sum())
print('----------------------------------------------------')
print('重复值检测结果：',Loan_data.duplicated().sum())

<<<<<<<<<<<<<<空值和重复值检测>>>>>>>>>>>>>
空值检测结果： 年龄       0
教育       0
工龄       0
地址       0
收入       0
负债率      0
信用卡负债    0
其他负债     0
违约       0
dtype: int64
----------------------------------------------------
重复值检测结果： 0


In [10]:
# ===== 高风险人群初探 =====
# 筛选年龄 > 50 且收入 > 50 的样本，按负债率排序，观察高负债人群特征
Loan_data_Debt_ratio = Loan_data.query('年龄 > 50 and 收入 > 50').sort_values('负债率')
Loan_data_Debt_ratio[['年龄','收入','负债率','违约']]

,年龄,收入,负债率,违约
292,51,159,2.8,0
258,51,82,4.4,0
506,52,89,4.7,0
270,51,120,7.6,0
475,52,76,7.7,0
557,52,234,7.7,0
528,51,249,7.8,0
630,53,61,7.9,0
569,54,114,8.5,0
114,52,64,8.6,0


In [11]:
# ===== 教育程度分组分析 =====
# agg 同时计算：平均收入、平均负债率、违约人数（违约=1 的计数）
Loan_data_Education_grouping = Loan_data.groupby('教育').agg(
    平均收入=('收入','mean'),
    平均负债率=('负债率','mean'),
    违约人数=('违约',lambda x:(x == 1).sum())
).round(2)
Loan_data_Education_grouping

,平均收入,平均负债率,违约人数
教育,,,
1,39.45,10.30,79
2,46.44,9.89,59
3,60.20,10.84,30
4,58.68,10.96,14
5,116.60,6.86,1


In [12]:
# ===== 年龄段划分 =====
# pd.cut 将年龄分为 20-30/30-40/40-50/50-60 四段
Loan_data['年龄段'] = pd.cut(Loan_data['年龄'],bins=[20,30,40,50,60],right=False)
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段
0,41,3,17,12,176,9.3,11.359392,5.008608,1,"[40, 50)"
1,27,1,10,6,31,17.3,1.362202,4.000798,0,"[20, 30)"
2,40,1,15,14,55,5.5,0.856075,2.168925,0,"[40, 50)"
3,41,1,15,14,120,2.9,2.658720,0.821280,0,"[40, 50)"
4,24,2,2,0,28,17.3,1.787436,3.056564,1,"[20, 30)"
...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,1,"[30, 40)"
696,29,2,6,4,21,11.5,0.369495,2.045505,0,"[20, 30)"
697,33,1,15,3,32,7.6,0.491264,1.940736,0,"[30, 40)"
698,45,1,19,22,77,8.4,2.302608,4.165392,0,"[40, 50)"


In [13]:
# 按年龄段统计信用卡负债均值
Loan_data_Education_grouping2 = Loan_data.groupby('年龄段')['信用卡负债'].mean().reset_index()
Loan_data_Education_grouping2

,年龄段,信用卡负债
0,"[20, 30)",0.933544
1,"[30, 40)",1.452390
2,"[40, 50)",2.236306
3,"[50, 60)",3.011463


In [14]:
# ===== 负债收入比 =====
# 计算 信用卡负债/其他负债 + 收入 作为负债压力指标
# ⚠️ 公式语义上较可疑（比例加收入量纲混乱），建议改为 总负债/收入
Loan_data['负债收入比'] =  (Loan_data['信用卡负债']/Loan_data['其他负债']) + Loan_data['收入']
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比
0,41,3,17,12,176,9.3,11.359392,5.008608,1,"[40, 50)",178.267974
1,27,1,10,6,31,17.3,1.362202,4.000798,0,"[20, 30)",31.340483
2,40,1,15,14,55,5.5,0.856075,2.168925,0,"[40, 50)",55.394700
3,41,1,15,14,120,2.9,2.658720,0.821280,0,"[40, 50)",123.237288
4,24,2,2,0,28,17.3,1.787436,3.056564,1,"[20, 30)",28.584786
...,...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,1,"[30, 40)",27.267427
696,29,2,6,4,21,11.5,0.369495,2.045505,0,"[20, 30)",21.180638
697,33,1,15,3,32,7.6,0.491264,1.940736,0,"[30, 40)",32.253133
698,45,1,19,22,77,8.4,2.302608,4.165392,0,"[40, 50)",77.552795


In [15]:
# ===== 风险等级规则 =====
# 自定义函数：违约=1 或 负债率>0.5 判定为高风险
def voli_Risk(s):
    if s['违约'] == 1 or s['负债率'] > 0.5:
       return '高风险'
    else:
        return '低风险'

In [16]:
# apply 按行应用规则函数，新增'风险等级'列
Loan_data['风险等级'] = Loan_data[['违约','负债率']].apply(voli_Risk,axis=1)
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级
0,41,3,17,12,176,9.3,11.359392,5.008608,1,"[40, 50)",178.267974,高风险
1,27,1,10,6,31,17.3,1.362202,4.000798,0,"[20, 30)",31.340483,高风险
2,40,1,15,14,55,5.5,0.856075,2.168925,0,"[40, 50)",55.394700,高风险
3,41,1,15,14,120,2.9,2.658720,0.821280,0,"[40, 50)",123.237288,高风险
4,24,2,2,0,28,17.3,1.787436,3.056564,1,"[20, 30)",28.584786,高风险
...,...,...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,1,"[30, 40)",27.267427,高风险
696,29,2,6,4,21,11.5,0.369495,2.045505,0,"[20, 30)",21.180638,高风险
697,33,1,15,3,32,7.6,0.491264,1.940736,0,"[30, 40)",32.253133,高风险
698,45,1,19,22,77,8.4,2.302608,4.165392,0,"[40, 50)",77.552795,高风险


In [17]:
# 统计高风险/低风险样本数量
print('<<<<<---------------------风险统计----------------------->>>>>')
print('高风险对象总数：',Loan_data.query('风险等级 == "高风险"')['风险等级'].count())
print('低风险对象总数：',Loan_data.query('风险等级 == "低风险"')['风险等级'].count())

<<<<<---------------------风险统计----------------------->>>>>
高风险对象总数： 699
低风险对象总数： 1


In [18]:
# ===== 违约预测规则 =====
# 规则：收入<30 且 负债率>15 预测为'会违约'（⚠️ 负债率>15 阈值偏大，实际负债率通常 0-1，此规则可能永远不触发）
def Predictive_model(s):
    if s['收入']<30 and s['负债率']>15:
        return '会违约'
    else:
        return '不会违约'

In [19]:
# 应用预测规则，新增'预测违约'列
Loan_data['预测违约'] = Loan_data[['收入','负债率']].apply(Predictive_model,axis=1)
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
0,41,3,17,12,176,9.3,11.359392,5.008608,1,"[40, 50)",178.267974,高风险,不会违约
1,27,1,10,6,31,17.3,1.362202,4.000798,0,"[20, 30)",31.340483,高风险,不会违约
2,40,1,15,14,55,5.5,0.856075,2.168925,0,"[40, 50)",55.394700,高风险,不会违约
3,41,1,15,14,120,2.9,2.658720,0.821280,0,"[40, 50)",123.237288,高风险,不会违约
4,24,2,2,0,28,17.3,1.787436,3.056564,1,"[20, 30)",28.584786,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,1,"[30, 40)",27.267427,高风险,不会违约
696,29,2,6,4,21,11.5,0.369495,2.045505,0,"[20, 30)",21.180638,高风险,不会违约
697,33,1,15,3,32,7.6,0.491264,1.940736,0,"[30, 40)",32.253133,高风险,不会违约
698,45,1,19,22,77,8.4,2.302608,4.165392,0,"[40, 50)",77.552795,高风险,不会违约


In [20]:
# 将违约列从 0/1 映射为 是/否，便于阅读
Loan_data['违约'] = Loan_data['违约'].astype(int)
Loan_data['违约'] = Loan_data['违约'].map({1:'是',0:'否'})
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
0,41,3,17,12,176,9.3,11.359392,5.008608,是,"[40, 50)",178.267974,高风险,不会违约
1,27,1,10,6,31,17.3,1.362202,4.000798,否,"[20, 30)",31.340483,高风险,不会违约
2,40,1,15,14,55,5.5,0.856075,2.168925,否,"[40, 50)",55.394700,高风险,不会违约
3,41,1,15,14,120,2.9,2.658720,0.821280,否,"[40, 50)",123.237288,高风险,不会违约
4,24,2,2,0,28,17.3,1.787436,3.056564,是,"[20, 30)",28.584786,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,是,"[30, 40)",27.267427,高风险,不会违约
696,29,2,6,4,21,11.5,0.369495,2.045505,否,"[20, 30)",21.180638,高风险,不会违约
697,33,1,15,3,32,7.6,0.491264,1.940736,否,"[30, 40)",32.253133,高风险,不会违约
698,45,1,19,22,77,8.4,2.302608,4.165392,否,"[40, 50)",77.552795,高风险,不会违约


In [21]:
# ===== 模型评估 =====
# ⚠️ 逻辑 bug：违约列已映射为'是/否'，而预测列为'会违约/不会违约'，字符串永远不相等
# 导致正确率恒为 0%，需统一两列取值后再比较（如都映射为 0/1）
print('=========================预测模型的评估===========================')
print('已违约人数:\n',Loan_data.query('违约 == "是"')['违约'].count())
print('模型认为会违约的人数:\n',Loan_data.query('预测违约 == "会违约"')['预测违约'].count())
print('=========================预测模型的正确率评估===========================')
correct = (Loan_data['违约'] == Loan_data['预测违约']).sum()
total = len(Loan_data)
accuracy = correct / total

print(f"预测正确数：{correct}")
print(f"总样本数：{total}")
print(f"正确率：{accuracy:.2%}")

=========================预测模型的评估===========================
已违约人数:
 183
模型认为会违约的人数:
 58
=========================预测模型的正确率评估===========================
预测正确数：0
总样本数：700
正确率：0.00%


In [22]:
# 查看样本总数
len(Loan_data['违约'])

700

In [23]:
# ⚠️ 无意义比较：Series == int，且比较对象错误（应为 违约列 与 预测违约列 逐元素比较）
Loan_data['违约'] == Loan_data['预测违约'].sum()

0      False
1      False
2      False
3      False
4      False
       ...  
695    False
696    False
697    False
698    False
699    False
Name: 违约, Length: 700, dtype: bool

In [24]:
# ===== 透视表：教育 × 违约 =====
# 行=教育，列=违约，值=收入，同时计算 sum 与 mean
Loan_data_pivot_table = pd.pivot_table(Loan_data,values='收入',index='教育',columns='违约',aggfunc=['sum','mean'])
Loan_data_pivot_table

sum              mean           
违约      否     是           否          是
教育                                    
1   12090  2585   41.262799  32.721519
2    7283  1913   52.395683  32.423729
3    3202  2035   56.175439  67.833333
4    1291   939   53.791667  67.071429
5     513    70  128.250000  70.000000

In [25]:
# 需求说明：按教育为行、违约为列，统计平均收入
按 教育 为行，违约 为列，统计 平均收入。

SyntaxError: invalid character '，' (U+FF0C) (115771273.py, line 1)

In [26]:
# 透视表：平均收入（mean）
Loan_data_pivot_table_Average_income = pd.pivot_table(Loan_data,values='收入',index='教育',columns='违约',aggfunc='mean')
Loan_data_pivot_table_Average_income

违约,否,是
教育,,
1,41.262799,32.721519
2,52.395683,32.423729
3,56.175439,67.833333
4,53.791667,67.071429
5,128.250000,70.000000


In [27]:
# 需求说明：同上，并将空缺值填充为 0
按 教育 为行，违约 为列，统计 平均收入，并将空缺值填充为 0

SyntaxError: invalid character '，' (U+FF0C) (609487072.py, line 1)

In [28]:
# 透视表：平均收入，fill_value=0 填充空值
pd.pivot_table(Loan_data, values='收入', index='教育', columns='违约', aggfunc='mean', fill_value=0)

违约,否,是
教育,,
1,41.262799,32.721519
2,52.395683,32.423729
3,56.175439,67.833333
4,53.791667,67.071429
5,128.250000,70.000000


In [29]:
# 透视表：同时输出 mean 和 sum
Loan_data_pivot_table_Average_income2 = pd.pivot_table(Loan_data,values='收入',index='教育',columns='违约',aggfunc=['mean','sum'])
Loan_data_pivot_table_Average_income2

mean               sum      
违约           否          是      否     是
教育                                    
1    41.262799  32.721519  12090  2585
2    52.395683  32.423729   7283  1913
3    56.175439  67.833333   3202  2035
4    53.791667  67.071429   1291   939
5   128.250000  70.000000    513    70

In [30]:
# 透视表：多值列（收入、负债率）同时统计
Loan_data_pivot_table_Debt_ratio = pd.pivot_table(Loan_data,values=['收入','负债率'],index='教育',columns='违约',aggfunc='mean')
Loan_data_pivot_table_Debt_ratio

收入                  负债率           
违约           否          是         否          是
教育                                            
1    41.262799  32.721519  8.875427  15.569620
2    52.395683  32.423729  8.333813  13.549153
3    56.175439  67.833333  8.536842  15.226667
4    53.791667  67.071429  9.033333  14.257143
5   128.250000  70.000000  6.225000   9.400000

In [31]:
# 透视表：索引换为[违约, 教育]双层，统计平均收入
Loan_data_pivot_table_Debt_ratio2 = pd.pivot_table(Loan_data,values='收入',index=['违约','教育'],aggfunc='mean')
Loan_data_pivot_table_Debt_ratio2

收入
违约 教育            
否  1    41.262799
   2    52.395683
   3    56.175439
   4    53.791667
   5   128.250000
是  1    32.721519
   2    32.423729
   3    67.833333
   4    67.071429
   5    70.000000

In [32]:
# 透视表：margins=True 增加总计行列
pd.pivot_table(Loan_data,values='收入',index='教育',columns='违约',aggfunc='mean',margins=True,margins_name='总计')

违约,否,是,总计
教育,,,
1,41.262799,32.721519,39.448925
2,52.395683,32.423729,46.444444
3,56.175439,67.833333,60.195402
4,53.791667,67.071429,58.684211
5,128.250000,70.000000,116.600000
总计,47.154739,41.213115,45.601429


In [33]:
# 透视表：aggfunc='size' 统计各分组样本数（频数矩阵）
pd.pivot_table(Loan_data,index='教育',columns='违约',aggfunc='size',fill_value=0)

违约,否,是
教育,,
1,293,79
2,139,59
3,57,30
4,24,14
5,4,1


In [34]:
# 查看当前数据
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
0,41,3,17,12,176,9.3,11.359392,5.008608,是,"[40, 50)",178.267974,高风险,不会违约
1,27,1,10,6,31,17.3,1.362202,4.000798,否,"[20, 30)",31.340483,高风险,不会违约
2,40,1,15,14,55,5.5,0.856075,2.168925,否,"[40, 50)",55.394700,高风险,不会违约
3,41,1,15,14,120,2.9,2.658720,0.821280,否,"[40, 50)",123.237288,高风险,不会违约
4,24,2,2,0,28,17.3,1.787436,3.056564,是,"[20, 30)",28.584786,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,是,"[30, 40)",27.267427,高风险,不会违约
696,29,2,6,4,21,11.5,0.369495,2.045505,否,"[20, 30)",21.180638,高风险,不会违约
697,33,1,15,3,32,7.6,0.491264,1.940736,否,"[30, 40)",32.253133,高风险,不会违约
698,45,1,19,22,77,8.4,2.302608,4.165392,否,"[40, 50)",77.552795,高风险,不会违约


In [35]:
# 透视表：年龄段 × 违约 的信用卡负债均值
pd.pivot_table(Loan_data,values='信用卡负债',index='年龄段',columns='违约',aggfunc='mean',fill_value=0)

违约,否,是
年龄段,,
"[20, 30)",0.714314,1.297144
"[30, 40)",1.189114,2.443024
"[40, 50)",1.603772,4.904806
"[50, 60)",2.761684,3.610934


In [36]:
# 查看数据
Loan_data

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
0,41,3,17,12,176,9.3,11.359392,5.008608,是,"[40, 50)",178.267974,高风险,不会违约
1,27,1,10,6,31,17.3,1.362202,4.000798,否,"[20, 30)",31.340483,高风险,不会违约
2,40,1,15,14,55,5.5,0.856075,2.168925,否,"[40, 50)",55.394700,高风险,不会违约
3,41,1,15,14,120,2.9,2.658720,0.821280,否,"[40, 50)",123.237288,高风险,不会违约
4,24,2,2,0,28,17.3,1.787436,3.056564,是,"[20, 30)",28.584786,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.262062,0.979938,是,"[30, 40)",27.267427,高风险,不会违约
696,29,2,6,4,21,11.5,0.369495,2.045505,否,"[20, 30)",21.180638,高风险,不会违约
697,33,1,15,3,32,7.6,0.491264,1.940736,否,"[30, 40)",32.253133,高风险,不会违约
698,45,1,19,22,77,8.4,2.302608,4.165392,否,"[40, 50)",77.552795,高风险,不会违约


In [38]:
# copy()：复制 DataFrame，避免后续操作影响原数据
Loan_data_copy=Loan_data.copy()

In [41]:
# set_index('教育')：将教育列设为索引
Loan_data_copy2 = Loan_data_copy.set_index('教育')
Loan_data_copy2.index

Index([3, 1, 1, 1, 2, 2, 1, 1, 1, 1,
       ...
       2, 1, 1, 3, 2, 2, 2, 1, 1, 1],
      dtype='int64', name='教育', length=700)

In [46]:
# ⚠️ 索引为'教育'文本，loc[2] 应返回教育=2 的行；若教育为字符串则此处会 KeyError
Loan_data_copy2.loc[2]

,年龄,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
教育,,,,,,,,,,,,
2,24,2,0,28,17.3,1.787436,3.056564,是,"[20, 30)",28.584786,高风险,会违约
2,41,5,5,25,10.2,0.392700,2.157300,否,"[40, 50)",25.182033,高风险,不会违约
2,36,9,6,49,8.6,0.817516,3.396484,是,"[30, 40)",49.240695,高风险,不会违约
2,36,13,6,41,16.4,2.918216,3.805784,是,"[30, 40)",41.766784,高风险,不会违约
2,21,1,2,16,18.0,0.241920,2.638080,是,"[20, 30)",16.091703,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...
2,26,8,1,40,11.8,0.443680,4.276320,否,"[20, 30)",40.103753,高风险,不会违约
2,24,0,5,16,7.3,0.024528,1.143472,否,"[20, 30)",16.021450,高风险,不会违约
2,48,6,1,66,12.1,2.315940,5.670060,否,"[40, 50)",66.408451,高风险,不会违约


In [47]:
# reset_index()：将索引重置回普通列
Loan_data_copy3 = Loan_data_copy2.reset_index()

In [51]:
# 查看重置后的索引
Loan_data_copy3.index

RangeIndex(start=0, stop=700, step=1)

In [52]:
# 再次查看原索引（练习 set_index/reset_index 的区别）
Loan_data_copy2.index

Index([3, 1, 1, 1, 2, 2, 1, 1, 1, 1,
       ...
       2, 1, 1, 3, 2, 2, 2, 1, 1, 1],
      dtype='int64', name='教育', length=700)

In [54]:
# 多层索引：按[教育, 违约]双重索引
Loan_data_copy4 = Loan_data_copy.set_index(['教育','违约'])
Loan_data_copy4.index

MultiIndex([(3, '是'),
            (1, '否'),
            (1, '否'),
            (1, '否'),
            (2, '是'),
            (2, '否'),
            (1, '否'),
            (1, '否'),
            (1, '是'),
            (1, '否'),
            ...
            (2, '否'),
            (1, '否'),
            (1, '是'),
            (3, '否'),
            (2, '否'),
            (2, '是'),
            (2, '否'),
            (1, '否'),
            (1, '否'),
            (1, '否')],
           names=['教育', '违约'], length=700)

In [55]:
# 查看多层索引数据
Loan_data_copy4

年龄  工龄  地址   收入   负债率      信用卡负债      其他负债       年龄段       负债收入比 风险等级  \
教育 违约                                                                          
3  是   41  17  12  176   9.3  11.359392  5.008608  [40, 50)  178.267974  高风险   
1  否   27  10   6   31  17.3   1.362202  4.000798  [20, 30)   31.340483  高风险   
   否   40  15  14   55   5.5   0.856075  2.168925  [40, 50)   55.394700  高风险   
   否   41  15  14  120   2.9   2.658720  0.821280  [40, 50)  123.237288  高风险   
2  是   24   2   0   28  17.3   1.787436  3.056564  [20, 30)   28.584786  高风险   
...    ..  ..  ..  ...   ...        ...       ...       ...         ...  ...   
   是   36   6  15   27   4.6   0.262062  0.979938  [30, 40)   27.267427  高风险   
   否   29   6   4   21  11.5   0.369495  2.045505  [20, 30)   21.180638  高风险   
1  否   33  15   3   32   7.6   0.491264  1.940736  [30, 40)   32.253133  高风险   
   否   45  19  22   77   8.4   2.302608  4.165392  [40, 50)   77.552795  高风险   
   否   37  12  14   44  14.7   2.994684  3.473316  [30, 40)   44.862197  高风险   

       预测违约  
教育 违约        
3  是   不会违约  
1  否   不会违约  
   否   不会违约  
   否   不会违约  
2  是    会违约  
...     ...  
   是   不会违约  
   否   不会违约  
1  否   不会违约  
   否   不会违约  
   否   不会违约  

[700 rows x 11 columns]

In [60]:
# ⚠️ 多层索引 loc[1] 只给第一层值，可能 KeyError，需用元组 loc[('教育值','是')]
Loan_data_copy4.loc[1]

,年龄,工龄,地址,收入,负债率,信用卡负债,其他负债,年龄段,负债收入比,风险等级,预测违约
违约,,,,,,,,,,,
否,27,10,6,31,17.3,1.362202,4.000798,"[20, 30)",31.340483,高风险,不会违约
否,40,15,14,55,5.5,0.856075,2.168925,"[40, 50)",55.394700,高风险,不会违约
否,41,15,14,120,2.9,2.658720,0.821280,"[40, 50)",123.237288,高风险,不会违约
否,39,20,9,67,30.6,3.833874,16.668126,"[30, 40)",67.230012,高风险,不会违约
否,43,12,11,38,3.6,0.128592,1.239408,"[40, 50)",38.103753,高风险,不会违约
...,...,...,...,...,...,...,...,...,...,...,...
否,47,31,8,253,7.2,9.308376,8.907624,"[40, 50)",254.044990,高风险,不会违约
是,53,0,26,27,28.9,2.754459,5.048541,"[50, 60)",27.545595,高风险,会违约
否,33,15,3,32,7.6,0.491264,1.940736,"[30, 40)",32.253133,高风险,不会违约


In [63]:
# 查看索引名称与前 10 个索引值
print(Loan_data_copy4.index.names)
print(Loan_data_copy4.index[:10])

['教育', '违约']
MultiIndex([(3, '是'),
            (1, '否'),
            (1, '否'),
            (1, '否'),
            (2, '是'),
            (2, '否'),
            (1, '否'),
            (1, '否'),
            (1, '是'),
            (1, '否')],
           names=['教育', '违约'])


In [65]:
# 多层索引精确定位：loc[(1, '是')]（教育=1 且违约=是）
Loan_data_copy4.loc[(1, '是')]

C:\Users\MSN\AppData\Local\Temp\ipykernel_15888\666284732.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  Loan_data_copy4.loc[(1, '是')]


年龄  工龄  地址  收入   负债率     信用卡负债      其他负债       年龄段      负债收入比 风险等级  \
教育 违约                                                                       
1  是   24   3   4  19  24.4  1.358348  3.277652  [20, 30)  19.414427  高风险   
   是   26   0   0  14   7.5  0.302400  0.747600  [20, 30)  14.404494  高风险   
   是   34   2  11  25  12.6  0.573300  2.576700  [30, 40)  25.222494  高风险   
   是   46  16  18  52  12.9  3.032016  3.675984  [40, 50)  52.824818  高风险   
   是   37   1   3  24  15.1  1.801128  1.822872  [30, 40)  24.988072  高风险   
...    ..  ..  ..  ..   ...       ...       ...       ...        ...  ...   
   是   30   7   2  33  25.4  1.165098  7.216902  [30, 40)  33.161440  高风险   
   是   20   4   0  14   9.7  0.200984  1.157016  [20, 30)  14.173709  高风险   
   是   35   7   5  39  16.1  1.701609  4.577391  [30, 40)  39.371742  高风险   
   是   34  10   1  33  10.3  2.501664  0.897336  [30, 40)  35.787879  高风险   
   是   53   0  26  27  28.9  2.754459  5.048541  [50, 60)  27.545595  高风险   

       预测违约  
教育 违约        
1  是    会违约  
   是   不会违约  
   是   不会违约  
   是   不会违约  
   是    会违约  
...     ...  
   是   不会违约  
   是   不会违约  
   是   不会违约  
   是   不会违约  
   是    会违约  

[79 rows x 11 columns]

In [68]:
# reset_index(level='违约')：只重置第二层索引
Loan_data_copy4.reset_index(level='违约')

,违约,年龄,工龄,地址,收入,负债率,信用卡负债,其他负债,年龄段,负债收入比,风险等级,预测违约
教育,,,,,,,,,,,,
3,是,41,17,12,176,9.3,11.359392,5.008608,"[40, 50)",178.267974,高风险,不会违约
1,否,27,10,6,31,17.3,1.362202,4.000798,"[20, 30)",31.340483,高风险,不会违约
1,否,40,15,14,55,5.5,0.856075,2.168925,"[40, 50)",55.394700,高风险,不会违约
1,否,41,15,14,120,2.9,2.658720,0.821280,"[40, 50)",123.237288,高风险,不会违约
2,是,24,2,0,28,17.3,1.787436,3.056564,"[20, 30)",28.584786,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...
2,是,36,6,15,27,4.6,0.262062,0.979938,"[30, 40)",27.267427,高风险,不会违约
2,否,29,6,4,21,11.5,0.369495,2.045505,"[20, 30)",21.180638,高风险,不会违约
1,否,33,15,3,32,7.6,0.491264,1.940736,"[30, 40)",32.253133,高风险,不会违约


In [72]:
# 对'收入'列切片后重置索引
Loan_data_copy4['收入'][1:5].reset_index()

,教育,违约,收入
0,1,否,31
1,1,否,55
2,1,否,120
3,2,是,28


In [76]:
# iloc 位置切片：取第 10-20 行
Loan_data_copy.iloc[10:20,]

,年龄,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
10,27,1,0,1,16,1.7,0.182512,0.089488,否,"[20, 30)",18.039514,高风险,不会违约
11,25,1,4,0,23,5.2,0.252356,0.943644,否,"[20, 30)",23.267427,高风险,不会违约
12,52,1,24,14,64,10.0,3.929600,2.470400,否,"[50, 60)",65.590674,高风险,不会违约
13,37,1,6,9,29,16.3,1.715901,3.011099,否,"[30, 40)",29.569859,高风险,会违约
14,48,1,22,15,100,9.1,3.703700,5.396300,否,"[40, 50)",100.686341,高风险,不会违约
15,36,2,9,6,49,8.6,0.817516,3.396484,是,"[30, 40)",49.240695,高风险,不会违约
16,36,2,13,6,41,16.4,2.918216,3.805784,是,"[30, 40)",41.766784,高风险,不会违约
17,43,1,23,19,72,7.6,1.181952,4.290048,否,"[40, 50)",72.275510,高风险,不会违约
18,39,1,6,9,61,5.7,0.563274,2.913726,否,"[30, 40)",61.193317,高风险,不会违约
19,41,3,0,21,26,1.7,0.099008,0.342992,否,"[40, 50)",26.288660,高风险,不会违约


In [79]:
# iloc 位置切片：取前 5 行前 3 列
Loan_data_copy.iloc[0:5,0:3]

,年龄,教育,工龄
0,41,3,17
1,27,1,10
2,40,1,15
3,41,1,15
4,24,2,2


In [81]:
# iloc 步长切片：隔行取数据（练习 iloc 用法）
Loan_data_copy.iloc[::2][['收入','违约']]

,收入,违约
0,176,是
2,55,否
4,28,是
6,67,否
8,19,是
...,...,...
690,16,否
692,27,是
694,66,否
696,21,否


In [87]:
# set_index('年龄')：以年龄为索引
Loan_data_copy5 = Loan_data_copy.set_index('年龄')
Loan_data_copy5.index[:10]

Index([41, 27, 40, 41, 24, 41, 39, 43, 24, 36], dtype='int64', name='年龄')

In [88]:
# 查看年龄索引数据
Loan_data_copy5

,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
年龄,,,,,,,,,,,,
41,3,17,12,176,9.3,11.359392,5.008608,是,"[40, 50)",178.267974,高风险,不会违约
27,1,10,6,31,17.3,1.362202,4.000798,否,"[20, 30)",31.340483,高风险,不会违约
40,1,15,14,55,5.5,0.856075,2.168925,否,"[40, 50)",55.394700,高风险,不会违约
41,1,15,14,120,2.9,2.658720,0.821280,否,"[40, 50)",123.237288,高风险,不会违约
24,2,2,0,28,17.3,1.787436,3.056564,是,"[20, 30)",28.584786,高风险,会违约
...,...,...,...,...,...,...,...,...,...,...,...,...
36,2,6,15,27,4.6,0.262062,0.979938,是,"[30, 40)",27.267427,高风险,不会违约
29,2,6,4,21,11.5,0.369495,2.045505,否,"[20, 30)",21.180638,高风险,不会违约
33,1,15,3,32,7.6,0.491264,1.940736,否,"[30, 40)",32.253133,高风险,不会违约


In [89]:
# loc[30]：按索引定位年龄=30 的记录
Loan_data_copy5.loc[30]

,教育,工龄,地址,收入,负债率,信用卡负债,其他负债,违约,年龄段,负债收入比,风险等级,预测违约
年龄,,,,,,,,,,,,
30,1,1,10,22,10.5,1.138830,1.171170,否,"[30, 40)",22.972387,高风险,不会违约
30,2,10,4,22,16.1,1.409716,2.132284,否,"[30, 40)",22.661130,高风险,会违约
30,4,2,8,25,10.0,1.770000,0.730000,否,"[30, 40)",27.424658,高风险,不会违约
30,1,11,1,33,8.4,0.681912,2.090088,否,"[30, 40)",33.326260,高风险,不会违约
30,1,1,8,17,11.0,0.746130,1.123870,否,"[30, 40)",17.663894,高风险,不会违约
30,1,4,0,33,4.2,0.500346,0.885654,否,"[30, 40)",33.564945,高风险,不会违约
30,1,11,9,27,0.9,0.073872,0.169128,否,"[30, 40)",27.436782,高风险,不会违约
30,2,4,4,21,18.3,0.491904,3.351096,否,"[30, 40)",21.146789,高风险,会违约
30,1,10,11,39,9.5,2.056275,1.648725,否,"[30, 40)",40.247191,高风险,不会违约


In [93]:
# ⚠️ 对字符串标签做切片需要索引有序（本列为年龄数值），此写法会报错或返回空
Loan_data_copy.loc[:'年龄',:'负债率']

,年龄,教育,工龄,地址,收入,负债率
0,41,3,17,12,176,9.3
1,27,1,10,6,31,17.3
2,40,1,15,14,55,5.5
3,41,1,15,14,120,2.9
4,24,2,2,0,28,17.3
...,...,...,...,...,...,...
695,36,2,6,15,27,4.6
696,29,2,6,4,21,11.5
697,33,1,15,3,32,7.6
698,45,1,19,22,77,8.4


In [97]:
# query 条件筛选 + iloc 列切片组合使用
Loan_data_copy.query('收入 > 50').iloc[:,1:6]

,教育,工龄,地址,收入,负债率
0,3,17,12,176,9.3
2,1,15,14,55,5.5
3,1,15,14,120,2.9
6,1,20,9,67,30.6
12,1,24,14,64,10.0
...,...,...,...,...,...
684,1,18,10,53,10.5
688,1,12,12,68,10.8
691,1,31,8,253,7.2
694,2,6,1,66,12.1


In [100]:
# ⚠️ 语法错误：Loan_data_copy. 后面缺少属性/方法名，此 cell 无法运行
print(Loan_data_copy.loc[2,'年龄'])
print(Loan_data_copy.)

40
